# everymd examples

**Every document → Markdown that AI can use.** everymd converts one file or URL at a time. [Docling](https://github.com/docling-project/docling) does the conversion at full quality, then a **copy-editor agent** (LangChain Deep Agents, `gpt-6-luna` by default) checks it against the page image and fixes what the image clearly shows is wrong. Both read **one page at a time, like a person**: Docling converts a page, the copy-editor edits it, its page file is written and its memory is let go, then the next page.

```python
from everymd import convert
result = convert("path/or/url", resolution="high", use_gpu=True, llm_layer=True, batch_size=1)
```

Four controls, nothing more:

| Control | Values | Meaning |
|---|---|---|
| `resolution` | `"low"` · `"medium"` · `"high"` · `"max"` | How hard Docling works (image scale, table model, formula and code enrichment) |
| `use_gpu` | `True` · `False` | Use MPS/CUDA when available (Docker on a Mac has no GPU, so it runs on CPU) |
| `llm_layer` | `True` · `False` | Copy-editor on, or pure Docling output |
| `batch_size` | `1` · `n` · `None` | Pages per copy-editor pass: one, n, or all at once |

Each example writes to `samples/outputs/<name>/`:

| File | What it is |
|---|---|
| `<name>.md` | The whole document, with front-matter and invisible page anchors |
| `<name>.p0001.md`, … | One file per page (`.s0001.md` per slide), the same text as in the whole document |
| `<name>.brief.md` | What the document is, a summary with page references, key terms (copy-editor on) |
| `<name>.index.md` | Which pages contain what (documents with pages or slides, copy-editor on) |
| `images/` | Pictures, and a screenshot of every table linked under it |
| `<name>.docling.json`, `report.json` | The lossless Docling document; what happened, page notes, tokens and cost |

A single image gets an `ai_caption` in its front-matter instead of a brief. The model is chosen in `everymd/models.py` or with `EVERYMD_MODEL` in `.env` (see the last section). Run this notebook inside the Docker container (see the README).

In [1]:
import logging
import os
import tempfile
import warnings
from pathlib import Path

# Keep the output readable: hide third-party progress bars and start-up logs.
warnings.filterwarnings("ignore", message="IProgress not found")
logging.getLogger("RapidOCR").disabled = True
from transformers.utils import logging as transformers_logging

transformers_logging.set_verbosity_error()
transformers_logging.disable_progress_bar()

from dotenv import load_dotenv

from everymd import UnsupportedInput, convert, models

load_dotenv()

# Settings used by every cell below.
RESOLUTION = "high"
USE_GPU = True
BATCH_SIZE = 1

# Test runs use low reasoning effort (the owner's rule). Delete this line for production (default: medium).
os.environ.setdefault("EVERYMD_REASONING_EFFORT", "low")

try:
    models.build()  # checks the key and package for the model chosen in models.py or EVERYMD_MODEL
    LLM_LAYER = True
    print("Copy-editor model:", models.model_id())
except RuntimeError as error:
    LLM_LAYER = False
    print(f"Copy-editor off ({error}). Showing pure Docling output.")

INPUTS = Path("samples/inputs")
OUT = Path("samples/outputs")


def run(source, chars=1500, **overrides):
    """Convert with the settings above, then print a short summary and the start of the Markdown."""
    options = dict(output_dir=OUT, resolution=RESOLUTION, use_gpu=USE_GPU, llm_layer=LLM_LAYER, batch_size=BATCH_SIZE)
    options.update(overrides)
    result = convert(source, **options)
    report = result.report
    cost = report["estimated_cost_usd"]
    print(f"Detected type: {report['detected_type']} (by {report['detection']['method']})")
    print(f"Pages: {len(report['pages'])}   Time: {report['seconds']} s   "
          f"Estimated cost: {'n/a' if cost is None else f'${cost:.4f}'}")
    for note in report["notes"]:
        print("Note:", note)
    print("Markdown:  ", result.markdown_path)
    if result.page_paths:
        print("Page files:", len(result.page_paths), f"({result.page_paths[0].name}, ...)")
    for label, path in (("Brief:     ", result.brief_path), ("Index:     ", result.index_path)):
        if path:
            print(label, path)
    if report["caption"]:
        print("Caption:   ", report["caption"])
    print("Report:    ", result.report_path)
    if chars:
        print("\n" + result.markdown[:chars] + ("\n[...]" if len(result.markdown) > chars else ""))
    return result


def show(path):
    """Print a file the conversion wrote, without its front-matter."""
    print(Path(path).read_text(encoding="utf-8").split("\n---\n", 1)[-1].strip())

Copy-editor model: openai:gpt-6-luna


## Digital PDF

A two-page report with a table, a bar-chart image, a formula and running headers.

In [2]:
report = run(INPUTS / "report.pdf", chars=2500)

Detected type: pdf (by magika)
Pages: 2   Time: 31.7 s   Estimated cost: $0.0020
Markdown:   samples/outputs/report/report.md
Page files: 2 (report.p0001.md, ...)
Brief:      samples/outputs/report/report.brief.md
Index:      samples/outputs/report/report.index.md
Report:     samples/outputs/report/report.json

---
title: "Quarterly Field Report: Water Sensor Pilot"
source: "samples/inputs/report.pdf"
detected_type: "pdf"
pages: 2
converted_at: "2026-10-04T05:25:29+00:00"
converter: {"everymd": "0.1.0", "docling": "2.132.0"}
resolution: "high"
ai_copy_edited: true
ai_model: "gpt-6-luna"
---

<!-- page: 1 -->

# Quarterly Field Report: Water Sensor Pilot

This report summarises the second quarter of the community water sensor pilot. Volunteers installed soil moisture sensors in four regions and logged how many stayed online.

## 1. Results by region

| Region   |   Q1 sensors |   Q2 sensors | Change   |
|----------|--------------|--------------|----------|
| North    |          120 |   

## Reading page by page: continuity, brief and index

A five-page log where every page break cuts something in half: a table runs onto page 2 without its header row, a sentence (and the word *irriga-tion*) breaks across pages 2 and 3, a numbered list runs from page 3 to page 4, and a chart on page 4 has its caption on page 5. Each page file holds exactly what is printed on its page: page 2 ends `irriga-` and page 3 starts `tion schedule changed`, so nothing is duplicated.

After each page the copy-editor writes a **continuity note** (where it is, what is still open) for the next page, and a **page note** (what the page covers). After the last page, one more call turns the page notes into the **brief** and the **index**.

In [3]:
log = run(INPUTS / "continuity.pdf", chars=0)
print("\nContinuity notes:")
for page in log.report["pages"]:
    print(f"\n  after page {page['number']}:\n    " + (page["continuity_note"] or "None").replace("\n", "\n    "))

Detected type: pdf (by magika)
Pages: 5   Time: 50.3 s   Estimated cost: $0.0039
Markdown:   samples/outputs/continuity/continuity.md
Page files: 5 (continuity.p0001.md, ...)
Brief:      samples/outputs/continuity/continuity.brief.md
Index:      samples/outputs/continuity/continuity.index.md
Report:     samples/outputs/continuity/report.json

Continuity notes:

  after page 1:
    Where I am: # Sensor Field Log > ## 1 Readings.
    Still open: None.
    Watch for: Next page follows the readings section.

  after page 2:
    Where I am: # Sensor Field Log > ## 1 Readings.
    Still open: Table with columns Site, Date, Moisture (%), Battery (V), started page 1; last page-body text ends with the fragment “irriga-”.
    Watch for: Continue the cut-off word/sentence from “irriga-”; readings table may continue.

  after page 3:
    Where I am: ## 2 Maintenance steps.
    Still open: None.
    Watch for: Next page may continue the maintenance steps or begin another section.

  after page 4:
 

Page 2 as its own file: the table gets its column names back, marked as continued from page 1.

In [4]:
show(log.page_paths[1])

<!-- page: 2 -->

*(table continued from page 1)*

| Site | Date | Moisture (%) | Battery (V) |
|---|---|---:|---:|
| S29 | 2026-04-02 | 18 | 3.7 |
| S30 | 2026-04-03 | 25 | 3.3 |
| S31 | 2026-04-04 | 32 | 3.4 |
| S32 | 2026-04-05 | 39 | 3.5 |
| S33 | 2026-04-06 | 46 | 3.6 |
| S34 | 2026-04-07 | 53 | 3.7 |
| S35 | 2026-04-08 | 20 | 3.3 |
| S36 | 2026-04-09 | 27 | 3.4 |
| S37 | 2026-04-10 | 34 | 3.5 |
| S38 | 2026-04-11 | 41 | 3.6 |
| S39 | 2026-04-12 | 48 | 3.7 |
| S40 | 2026-04-13 | 15 | 3.3 |

![Table screenshot](images/table-p0002-1.png)

These readings come from the spring survey. Volunteers checked every site twice and logged the lower value. Sites in the river valley stayed wetter than the hill sites, as in earlier years. All readings above were taken before the irriga-

<!-- ai-edited: Added the continued table's column headers and continuation marker. -->


The brief (what the document says) and the index (where to find things). A RAG system can read the index, then open only the page files it needs.

In [5]:
show(log.brief_path)
print("\n" + "-" * 80 + "\n")
show(log.index_path)

# Brief: Sensor Field Log
## What this is
A spring-survey field log recording soil moisture and battery voltage at sensor sites, with maintenance steps and a recommendation for a follow-up survey. The running header identifies it as “Sensor Field Log - spring survey”; the readings are dated 2026-04-01 through 2026-04-28.
## Summary
- Section 1 gives the spring survey readings table, with dates, soil moisture (%) and battery (V) for sites S01–S28 (p. 1) and S29–S40 (p. 2). The accompanying account says volunteers checked every site twice and logged the lower value; river valley sites stayed wetter than hill sites, as in earlier years. The text continues on p. 3, noting that the irrigation schedule changed in May and dry readings at sites S31 to S40 need no follow-up (pp. 2–3).
- Section 2 lists actions for every site visit, including replacing the battery if it reads below 3.4 V; the ordered list continues with items 4–5 on p. 4 (pp. 3–4). A chart compares sensors online by region—North

## Scanned PDF

The same report printed, slightly rotated and scanned. OCR misreads a few things; the copy-editor fixes them from the page image.

In [6]:
scan = run(INPUTS / "scanned_report.pdf", chars=2500)

Detected type: pdf (by magika)
Pages: 2   Time: 40.8 s   Estimated cost: $0.0019
Markdown:   samples/outputs/scanned_report/scanned_report.md
Page files: 2 (scanned_report.p0001.md, ...)
Brief:      samples/outputs/scanned_report/scanned_report.brief.md
Index:      samples/outputs/scanned_report/scanned_report.index.md
Report:     samples/outputs/scanned_report/report.json

---
title: "Quarterly Field Report: Water Sensor Pilot"
source: "samples/inputs/scanned_report.pdf"
detected_type: "pdf"
pages: 2
converted_at: "2026-10-04T05:26:59+00:00"
converter: {"everymd": "0.1.0", "docling": "2.132.0"}
resolution: "high"
ai_copy_edited: true
ai_model: "gpt-6-luna"
---

<!-- page: 1 -->

# Quarterly Field Report: Water Sensor Pilot

This report summarises the second quarter of the community water sensor pilot. Volunteers installed soil moisture sensors in four regions and logged how many stayed online.

## 1. Results by region

| Region   |   Q1 sensors |   Q2 sensors | Change   |
|----------|

### Scanned PDF, Docling only (for comparison)

The same scan with `llm_layer=False`. Compare the formula and the chart with the cell above.

In [7]:
scan_plain = run(INPUTS / "scanned_report.pdf", chars=2500, llm_layer=False, output_dir=OUT / "docling-only")

Detected type: pdf (by magika)
Pages: 2   Time: 18.2 s   Estimated cost: $0.0000
Markdown:   samples/outputs/docling-only/scanned_report/scanned_report.md
Page files: 2 (scanned_report.p0001.md, ...)
Report:     samples/outputs/docling-only/scanned_report/report.json

---
title: "Quarterly Field Report: Water Sensor Pilot"
source: "samples/inputs/scanned_report.pdf"
detected_type: "pdf"
pages: 2
converted_at: "2026-10-04T05:27:24+00:00"
converter: {"everymd": "0.1.0", "docling": "2.132.0"}
resolution: "high"
ai_copy_edited: false
ai_model: null
---

<!-- page: 1 -->

## Quarterly Field Report: Water Sensor Pilot

This report summarises the second quarter of the community water sensor pilot. Volunteers installed soil moisture sensors in four regions and logged how many stayed online.

## 1. Results by region

| Region   |   Q1 sensors |   Q2 sensors | Change   |
|----------|--------------|--------------|----------|
| North    |          120 |          150 | +25.0%   |
| South    |      

## Image

A photographed-style invoice (PNG): OCR plus table structure. A single image gets a one-sentence caption (`ai_caption` in the front-matter) instead of a brief or index.

In [8]:
invoice = run(INPUTS / "invoice.png", chars=2000)

Detected type: image (by magika)
Pages: 1   Time: 12.0 s   Estimated cost: $0.0009
Markdown:   samples/outputs/invoice/invoice.md
Page files: 1 (invoice.p0001.md, ...)
Caption:    Invoice INV-2026-0142 dated 15 September 2026, billed to Riverside Community Garden. Includes an itemized table for soil moisture sensors, solar chargers, and installation, totals of 558.00 subtotal, 19% VAT (106.02), and 664.02 total due (EUR), plus payment terms and supplier address.
Report:     samples/outputs/invoice/report.json

---
title: "INVOICE"
source: "samples/inputs/invoice.png"
detected_type: "image"
pages: 1
converted_at: "2026-10-04T05:27:36+00:00"
converter: {"everymd": "0.1.0", "docling": "2.132.0"}
resolution: "high"
ai_copy_edited: true
ai_model: "gpt-6-luna"
ai_caption: "Invoice INV-2026-0142 dated 15 September 2026, billed to Riverside Community Garden. Includes an itemized table for soil moisture sensors, solar chargers, and installation, totals of 558.00 subtotal, 19% VAT (106.02), and 

## Word

A proposal with headings, bullets, a table and an image. LibreOffice renders it to PDF first, so the copy-editor sees real pages.

In [9]:
proposal = run(INPUTS / "proposal.docx", chars=2000)

Detected type: docx (by magika)
Pages: 2   Time: 24.3 s   Estimated cost: $0.0017
Markdown:   samples/outputs/proposal/proposal.md
Page files: 2 (proposal.p0001.md, ...)
Brief:      samples/outputs/proposal/proposal.brief.md
Index:      samples/outputs/proposal/proposal.index.md
Report:     samples/outputs/proposal/report.json

---
title: "Project Proposal: Community Sensor Network"
source: "samples/inputs/proposal.docx"
detected_type: "docx"
pages: 2
converted_at: "2026-10-04T05:27:54+00:00"
converter: {"everymd": "0.1.0", "docling": "2.132.0"}
resolution: "high"
ai_copy_edited: true
ai_model: "gpt-6-luna"
---

<!-- page: 1 -->

# Project Proposal: Community Sensor Network

## Background

Local growers lose crops every summer because they cannot see when soil dries out. This proposal extends the water sensor pilot into a permanent community network.

## Goals

- Keep at least 90% of sensors online through the year.
- Send growers a text message when soil moisture drops below 20%.
- Sh

## PowerPoint

Three slides with a table, a chart and speaker notes. Rendering drops notes, so everymd reads them from the file and attaches them to their slide.

In [10]:
deck = run(INPUTS / "board_deck.pptx", chars=2500)

RapidOCR returned empty result!


Detected type: pptx (by magika)
Pages: 3   Time: 24.9 s   Estimated cost: $0.0020
Markdown:   samples/outputs/board_deck/board_deck.md
Page files: 3 (board_deck.s0001.md, ...)
Brief:      samples/outputs/board_deck/board_deck.brief.md
Index:      samples/outputs/board_deck/board_deck.index.md
Report:     samples/outputs/board_deck/report.json

---
title: "Board Update: Q2 2026"
source: "samples/inputs/board_deck.pptx"
detected_type: "pptx"
pages: 3
converted_at: "2026-10-04T05:28:20+00:00"
converter: {"everymd": "0.1.0", "docling": "2.132.0"}
resolution: "high"
ai_copy_edited: true
ai_model: "gpt-6-luna"
---

<!-- slide: 1 -->

## Board Update: Q2 2026

Community water sensor pilot

**Speaker notes:** Welcome the board. The pilot finished its second quarter on schedule.

<!-- slide: 2 -->

## Sensor rollout by region

| Region   |   Q1 sensors |   Q2 sensors |
|----------|--------------|--------------|
| North    |          120 |          150 |
| South    |           95 |           88 

## HTML file

A web page with navigation, a cookie banner and a footer around the real content. Chromium prints it to PDF first.

In [11]:
page = run(INPUTS / "page.html", chars=2000)

Detected type: html (by magika)
Pages: 2   Time: 25.9 s   Estimated cost: $0.0017
Markdown:   samples/outputs/page/page.md
Page files: 2 (page.p0001.md, ...)
Brief:      samples/outputs/page/page.brief.md
Report:     samples/outputs/page/report.json

---
title: "Setting up a garden moisture sensor"
source: "samples/inputs/page.html"
detected_type: "html"
pages: 2
converted_at: "2026-10-04T05:28:45+00:00"
converter: {"everymd": "0.1.0", "docling": "2.132.0"}
resolution: "high"
ai_copy_edited: true
ai_model: "gpt-6-luna"
---

<!-- page: 1 -->

## Setting up a garden moisture sensor

![Image](images/image_000000_fc8faa28f0f73e96cf2eb6de44604b729e067085245be121fca97a7e45976763.png)

> **AI-generated description:** A grouped bar chart titled “Sensors online by region” compares Q1 2026 and Q2 2026. Values are North 120 and 150, South 95 and 88, East 130 and 162, and West 80 and 97, respectively. Q2 is higher in North, East, and West, while Q1 is higher in South; East has the highest values a

## Web URL

A live Wikipedia article ([Markdown](https://en.wikipedia.org/wiki/Markdown), CC BY-SA 4.0; see `samples/README.md`).

In [12]:
article = run("https://en.wikipedia.org/wiki/Markdown", chars=2500)

Detected type: url (by url)
Pages: 9   Time: 164.5 s   Estimated cost: $0.0147
Markdown:   samples/outputs/en.wikipedia.org-Markdown/en.wikipedia.org-Markdown.md
Page files: 9 (en.wikipedia.org-Markdown.p0001.md, ...)
Brief:      samples/outputs/en.wikipedia.org-Markdown/en.wikipedia.org-Markdown.brief.md
Report:     samples/outputs/en.wikipedia.org-Markdown/report.json

---
title: "Markdown"
source: "https://en.wikipedia.org/wiki/Markdown"
detected_type: "url"
pages: 9
converted_at: "2026-10-04T05:31:28+00:00"
converter: {"everymd": "0.1.0", "docling": "2.132.0"}
resolution: "high"
ai_copy_edited: true
ai_model: "gpt-6-luna"
---

<!-- page: 1 -->

![Image](images/image_000000_a8181d9eb22b3eae1a40dbe2d1d3072f18b1975a97e4e8f21c40a1f516cd78e8.png)

# Markdown

**Markdown**[9] is a [lightweight markup language](https://en.wikipedia.org/wiki/Lightweight_markup_language) for creating [formatted text](https://en.wikipedia.org/wiki/Formatted_text) using a [plain-text editor](https://en.wikipe

## EPUB

An e-book has no pages and no page images, and Docling reads its text straight from the file, so the copy-editor is skipped (nothing to check against). The brief is still written.

In [13]:
booklet = run(INPUTS / "booklet.epub")

Detected type: epub (by magika)
Pages: 1   Time: 4.8 s   Estimated cost: $0.0003
Markdown:   samples/outputs/booklet/booklet.md
Brief:      samples/outputs/booklet/booklet.brief.md
Report:     samples/outputs/booklet/report.json

---
title: "Chapter 1: Choosing a bed"
source: "samples/inputs/booklet.epub"
detected_type: "epub"
pages: null
converted_at: "2026-10-04T05:31:35+00:00"
converter: {"everymd": "0.1.0", "docling": "2.132.0"}
resolution: "high"
ai_copy_edited: false
ai_model: "gpt-6-luna"
---

# Chapter 1: Choosing a bed

Pick a spot with six hours of sun and soil that drains within a day of rain.

- Raised beds warm up faster in spring.
- Ground beds hold water longer in summer.

# Chapter 2: Watering

Water deeply and less often. A moisture sensor removes the guesswork: water when it reads below 20%.

Morning watering loses the least to evaporation.



## Email

An `.eml` message with a plain-text body and an HTML alternative. Like an e-book, it has no page images, so the copy-editor is skipped and the brief is still written.

In [14]:
email = run(INPUTS / "update.eml")

Detected type: email (by magika)
Pages: 1   Time: 3.7 s   Estimated cost: $0.0003
Markdown:   samples/outputs/update/update.md
Brief:      samples/outputs/update/update.brief.md
Report:     samples/outputs/update/report.json

---
title: "Pilot update: 497 sensors online"
source: "samples/inputs/update.eml"
detected_type: "email"
pages: null
converted_at: "2026-10-04T05:31:40+00:00"
converter: {"everymd": "0.1.0", "docling": "2.132.0"}
resolution: "high"
ai_copy_edited: false
ai_model: "gpt-6-luna"
---

# Pilot update: 497 sensors online

From: Priya Nair &lt;priya@example.org&gt;

To: Pilot volunteers &lt;volunteers@example.org&gt;

Date: 2026-09-30T09:15:00+00:00

Hi all,

Quick update on the water sensor pilot:

- 497 sensors are online, up from 425 last quarter.  
- South lost 7 sensors in the June storms; replacements arrive next week.  
- The Q3 budget (EUR 2,735) goes to the steering group on Monday.

Thanks for all the weekend installs!  
Priya



## Spreadsheets are rejected

Excel files are not supported in this version, and everymd says so clearly.

In [15]:
from openpyxl import Workbook

book = Workbook()
book.active.append(["Region", "Q1", "Q2"])
book.active.append(["North", 120, 150])
with tempfile.TemporaryDirectory() as folder:
    path = Path(folder) / "numbers.xlsx"
    book.save(path)
    try:
        convert(path)
    except UnsupportedInput as error:
        print("Rejected:", error)

Rejected: 'numbers.xlsx' is a spreadsheet. Spreadsheets (XLSX, XLS, ODS, CSV) are not supported in this version.


## Switching the model

Everything about the model lives in `everymd/models.py`. To use another provider, set it in `.env` (no code changes) and restart the kernel:

| `.env` | Model |
|---|---|
| *(nothing)* | `openai:gpt-6-luna`, the default |
| `EVERYMD_MODEL=anthropic:<model>` and `ANTHROPIC_API_KEY=...` | Anthropic |
| `EVERYMD_MODEL=google_genai:<model>` and `GOOGLE_API_KEY=...` | Google Gemini |
| `EVERYMD_MODEL=openai:<model>` and `EVERYMD_BASE_URL=http://host:8000/v1` | Any OpenAI-compatible server: vLLM, LM Studio, llama.cpp, OpenRouter, Together, Groq; open-source models included |

The model must support tool calling. For a model that can't read images, add `EVERYMD_MODEL_SEES_IMAGES=false`. You can also pass `model="provider:model"` to `convert()` for one call.